In [1]:
import os
from docx import Document
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json
import os
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

F:\Pytorch\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Architecture

# Folder Structure

# Template Structure

# LLM Client

In [2]:
try:
    import ollama
except ImportError:
    ollama = None


class LLMClient:
    """
    Generic LLM client with:
    - Ollama local testing mode
    - API-based production mode
    """

    def __init__(self, api_key=None, base_url=None, model=None, testing=True):

        self.api_key = api_key
        self.base_url = base_url.rstrip("/") if base_url else None
        self.model = model
        self.testing = testing

    # ----------------------------------------------------
    # MAIN ENTRY
    # ----------------------------------------------------

    def chat(self, prompt):

        if self.testing:
            return self._chat_ollama(prompt)

        return self._chat_api(prompt)

    # ----------------------------------------------------
    # LOCAL OLLAMA MODE
    # ----------------------------------------------------
    def generate(self, prompt):
        return self.chat(prompt)
    def _chat_ollama(self, prompt):

        if ollama is None:
            raise ImportError("ollama package not installed")

        response = ollama.chat(
            model=self.model or "llama3",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        return response["message"]["content"]

    # ----------------------------------------------------
    # API MODE (enterprise / internal LLM)
    # ----------------------------------------------------

    def _chat_api(self, prompt):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": self.model,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }

        response = requests.post(
            f"{self.base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        return data["choices"][0]["message"]["content"]

In [3]:
llm_client = LLMClient(
    testing=True,
    model="llama3"
)

## Step 1: Input arguments for:
* What type of GP
* Structure of GP

### 1.1 Input for GP

In [4]:
def detect_procedure_type(user_request):
    """
    Detects the operational procedure type from a user request.

    This function performs lightweight keyword-based classification
    to identify the intended industrial procedure category.

    Supported procedure types:
    - startup
    - shutdown
    - maintenance
    - inspection
    - general (fallback)

    Args:
        user_request (str):
            User query or procedure generation request.

    Returns:
        str:
            Detected procedure type.
    """

    text = user_request.lower()

    if "startup" in text:
        return "startup"
    if "shutdown" in text:
        return "shutdown"
    if "maintenance" in text:
        return "maintenance"
    if "inspection" in text:
        return "inspection"

    return "general"

### 1.2 Structure to GP

In [5]:
procedure_template = """
1. Objective
2. Scope
3. Preconditions
    3.1 Required Conditions
    3.2 Exceptions
4. Required Equipment
5. Safety Precautions
6. Procedure Steps
7. Validation Checks
8. Escalation Conditions
"""

## Step 2: Preparing documents for RAG

### Step 2.1 Parse logic from a word document

In [6]:
HEADINGS = {
    "rubrik 1",
    "rubrik 2",
    "rubrik 3",
    "rubrik 4",
    "heading 1",
    "heading 2",
    "heading 3",
    "heading 4",
}

SECTION_RE = re.compile(r"^([\d.]+)\s+(.*)")
SPACE_RE = re.compile(r"\s+")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def parse_word_document(filepath):
    doc = Document(filepath)
    filename = os.path.basename(filepath)

    sections = []
    current = None
    buffer = []

    paragraphs = [
        (p.text.strip(), SPACE_RE.sub(" ", p.style.name.lower()))
        for p in doc.paragraphs
        if p.text.strip()
    ]

    def flush():
        nonlocal buffer, current

        if not current:
            return

        raw = "\n".join(buffer)

        header = f"{current['section_id'] or ''} {current['title']}".strip()

        sections.append({
            "document": filename,
            **current,
            "raw_text": raw,
            "sentences": buffer,
            "section_text": f"{header}\n{raw}",
            "retrieval_text": f"{header} {raw}"
        })

        buffer = []

    for text, style in paragraphs:

        # 🔥 FIX 1: normalize style more aggressively
        style = style.strip().lower()

        if style in HEADINGS:

            flush()

            match = SECTION_RE.match(text)

            if match:
                sid, title = match.groups()
                level = sid.count(".") + 1
            else:
                sid, title, level = None, text, 1

            current = {
                "section_id": sid,
                "title": title,
                "level": level
            }

        else:
            buffer.append(text)

    flush()

    if sections:
        return sections

    fallback = " ".join(text for text, _ in paragraphs)
    fallback = " ".join(fallback.split()[:200])

    return [{
        "document": filename,
        "section_id": "0",
        "title": "Full Document (Auto Chunk)",
        "level": 0,
        "raw_text": fallback,
        "sentences": SENTENCE_RE.split(fallback),
        "section_text": f"0 Full Document (Auto Chunk)\n{fallback}",
        "retrieval_text": f"0 Full Document (Auto Chunk) {fallback}"
    }]

### Step 2.2 Load in documents

In [7]:
def load_all_documents(folder_path):
    """
    Loads and parses all .docx documents from a folder.

    This function:
    - Iterates through every .docx file in the provided folder
    - Parses each document using parse_word_document()
    - Aggregates all extracted sections into a single list

    Args:
        folder_path (str):
            Path to the folder containing Word documents.

    Returns:
        list[dict]:
            Combined list of parsed sections from all documents.
    """
    all_sections = []

    for file in os.listdir(folder_path):

        if file.endswith(".docx"):

            path = os.path.join(folder_path, file)
            sections = parse_word_document(path)

            all_sections.extend(sections)
    
    return all_sections

### Step 2.3 Build section index

In [8]:
def build_section_index(sections):
    """
    Builds a hybrid section-level retrieval index.

    Retrieval units are full document sections.

    The index combines:
    - semantic embeddings
    - TF-IDF lexical retrieval
    - section metadata

    Args:
        sections (list[dict]):
            Parsed document sections.

    Returns:
        dict:
            Hybrid retrieval system.
    """

    if not sections:
        raise ValueError("No sections found")

    ########################################################
    # RETRIEVAL TEXTS
    ########################################################

    section_texts = [
        s["retrieval_text"]
        for s in sections
    ]

    ########################################################
    # SEMANTIC EMBEDDINGS
    ########################################################

    section_embeddings = embedding_model.encode(
        section_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    ########################################################
    # LEXICAL INDEX
    ########################################################

    tfidf = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    tfidf_matrix = tfidf.fit_transform(section_texts)

    ########################################################
    # RETURN HYBRID INDEX
    ########################################################

    return {
        "sections": sections,
        "section_texts": section_texts,
        "section_embeddings": np.asarray(section_embeddings),
        "tfidf": tfidf,
        "tfidf_matrix": tfidf_matrix
    }

### Step 2.4 Retrieve the relevant sections based on prompt and desired structure

In [9]:
def retrieve_sections(
    query,
    retrieval_system,
    section_heading=None,
    k=10,
    alpha=0.7
):
    """
    Hybrid section retrieval using:
    - semantic embeddings
    - TF-IDF lexical matching
    """

    ########################################################
    # QUERY CONSTRUCTION
    ########################################################

    if section_heading:
        query = f"{query}\n{section_heading}"

    ########################################################
    # RETRIEVAL COMPONENTS
    ########################################################

    sections = retrieval_system["sections"]

    section_embeddings = retrieval_system["section_embeddings"]

    ########################################################
    # SEMANTIC SEARCH
    ########################################################

    query_emb = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    semantic_scores = section_embeddings @ query_emb

    ########################################################
    # LEXICAL SEARCH
    ########################################################

    tfidf_vec = retrieval_system["tfidf"].transform([query])

    lexical_scores = (
        tfidf_vec @ retrieval_system["tfidf_matrix"].T
    ).toarray()[0]

    ########################################################
    # SCORE NORMALIZATION
    ########################################################

    def normalize(x):
        rng = np.ptp(x)
        return np.zeros_like(x) if rng == 0 else (x - x.min()) / rng

    semantic_scores = normalize(semantic_scores)
    lexical_scores = normalize(lexical_scores)

    ########################################################
    # HYBRID SCORING
    ########################################################

    scores = (
        alpha * semantic_scores
        + (1 - alpha) * lexical_scores
    )

    ########################################################
    # TOP-K (WITHOUT FULL SORT)
    ########################################################
    if len(scores) == 0:
        return []

    k = min(k, len(scores))
    top_idx = np.argpartition(scores, -k)[-k:]

    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    ########################################################
    # RETURN RANKED SECTIONS
    ########################################################

    return [sections[i] for i in top_idx]

### Step 2.5 First LLM to turn content into a structured format

In [10]:
JSON_RE = re.compile(r"\[.*?\]", re.DOTALL)
CODEBLOCK_RE = re.compile(r"```(?:json)?|```", re.I)

REQUIRED_FIELDS = {
    "requirement_type",
    "operational_phase",
    "requirement_statement",
    "criticality"
}
def extract_requirements(section, llm_client):
    """
    Extract structured requirements from a single section.
    """

    prompt = (
        "You are an industrial operations analyst.\n\n"
        "Extract ONLY explicit operational requirements.\n\n"
        f"SECTION:\n{section['section_id']} {section['title']}\n\n"
        f"CONTENT:\n{section['raw_text']}\n\n"
        "Return JSON ONLY as a list of objects:\n"
        "[{\n"
        "  \"requirement_type\": \"...\",\n"
        "  \"operational_phase\": \"...\",\n"
        "  \"requirement_statement\": \"...\",\n"
        "  \"criticality\": \"...\"\n"
        "}]\n\n"
        "Rules:\n"
        "- No hallucinations\n"
        "- No summarization\n"
        "- Only explicit requirements\n"
    )

    content = llm_client.generate(prompt)

    ########################################################
    # CLEAN OUTPUT
    ########################################################

    content = CODEBLOCK_RE.sub("", content)

    match = JSON_RE.search(content)

    if not match:
        return []

    try:
        reqs = json.loads(match.group(0))
    except json.JSONDecodeError:
        return []

    ########################################################
    # VALIDATION + ENRICHMENT
    ########################################################

    base_meta = {
        "source_document": section.get("document"),
        "source_section_id": section.get("section_id"),
        "source_title": section.get("title")
    }

    cleaned = []

    for r in reqs:
        if not isinstance(r, dict):
            continue

        if not REQUIRED_FIELDS.issubset(r.keys()):
            continue

        cleaned.append({**r, **base_meta})

    return cleaned

### Step 2.6 Extracts the requirements by using LLM function 2.5

In [11]:
def build_requirement_database(sections, llm_client):
    """
    Extract requirements ONLY from retrieved sections.
    """

    return [
        req
        for section in sections
        if section.get("raw_text")
        for req in extract_requirements(section, llm_client)
    ]

### Step 2.7 Removes semantically similar text

In [12]:
def deduplicate_requirements(requirements, threshold=0.9):
    if not requirements:
        return []

    texts = [r["requirement_statement"] for r in requirements]

    emb = embedding_model.encode(texts, normalize_embeddings=True)

    kept = []

    for i, e in enumerate(emb):
        if all((e @ emb[j]) < threshold for j in kept):
            kept.append(i)

    return [requirements[i] for i in kept]

### Step 2.8 Formats the requirements into a strucutre that is then used for the LLM Generation of GP

In [13]:
def format_requirements(requirements):
    """
    Formats requirements into LLM-grounded context block.
    """

    blocks = []

    for i, r in enumerate(requirements, 1):

        blocks.append(
f"""--- REQUIREMENT {i} ---
Type: {r.get('requirement_type', 'unknown')}
Phase: {r.get('operational_phase', 'unknown')}
Criticality: {r.get('criticality', 'unknown')}
Statement: {r.get('requirement_statement', 'unknown')}
Source: {r.get('source_document', 'unknown')} | {r.get('source_section_id', 'UNMAPPED')} | {r.get('source_title', 'unknown')}
"""
        )

    return "\n".join(blocks)

## Step 4: Generate section with the help of RAG

### Step 4.1 Generates the text

In [14]:
# Optimized
def generate_section(section_name, sections, user_request, llm_client):
    """
    Generates a grounded General Procedure section using retrieved document sections.
    """

    ########################################################
    # GROUNDING CONTEXT (SECTION-BASED, NOT REQUIREMENTS)
    ########################################################

    section_text = "\n\n".join(
        f"""--- SECTION ---
Title: {s.get('title', 'unknown')}
ID: {s.get('section_id', 'unknown')}

Content:
{s.get('retrieval_text', s.get('raw_text', ''))}
"""
        for s in sections
    )

    ########################################################
    # PROVENANCE TRACKING (NEW)
    ########################################################

    source_sections = [
        {
            "document": s.get("document", "unknown"),
            "section_id": s.get("section_id"),
            "title": s.get("title"),
            "level": s.get("level")
        }
        for s in sections
    ]

    ########################################################
    # PROMPT
    ########################################################

    prompt = f"""You are generating a General Procedure section:
        A General Procedure should:
    - Preserve the operational workflow and safety logic
    - Remove machine-specific details, dimensions, and identifiers
    - Generalize specialized terminology when appropriate
    - Keep critical warnings and validation steps
    - Be reusable across similar equipment types.

Use ONLY the provided retrieved sections as grounding material.

User Request:
{user_request}

Target Section:
{section_name}

############################################################
RETRIEVED SECTIONS
############################################################
{section_text}

############################################################
RULES
############################################################
- Use ONLY retrieved information
- Do not introduce external knowledge
- Do not hallucinate missing steps
- Do not include metadata (IDs, source labels)
- Write only the General Procedure section content

############################################################
OUTPUT FORMAT
############################################################
- Clear procedural structure
- Logical step ordering
- Bullet points where appropriate

Return ONLY the section content.
"""

    ########################################################
    # OUTPUT (STRUCTURED)
    ########################################################

    return {
        "content": str(llm_client.generate(prompt)),
        "sources": source_sections
    }

### 4.2 Coordinater function dictating structure to the above function

In [15]:
def generate_general_procedure(
    user_request,
    retrieval_system,
    llm_client,
    procedure_template
):
    """
    Template-driven SECTION-RAG SOP generation with clean structured output.
    """

    import re

    procedure_type = detect_procedure_type(user_request)

    template_lines = [
        line.strip()
        for line in procedure_template.splitlines()
        if line.strip()
    ]

    generated_sections = []
    previous_sections_summary = ""

    for line in template_lines:

        clean_heading = re.sub(r"^\d+(\.\d+)*\s*", "", line).strip()

        query = f"{user_request} {clean_heading}"

        retrieved_sections = retrieve_sections(
            query=query,
            retrieval_system=retrieval_system,
            k=10
        )

        ########################################################
        # SECTION GENERATION (FIXED STRUCTURE USAGE)
        ########################################################

        section_obj = generate_section(
            section_name=clean_heading,
            sections=retrieved_sections,
            user_request=user_request,
            llm_client=llm_client
        )

        section_text = section_obj["content"]
        section_sources = section_obj["sources"]

        ########################################################
        # HEADING LEVEL LOGIC
        ########################################################

        level = line.count(".") + 1

        if level <= 1:
            heading = f"## {clean_heading}"
        elif level == 2:
            heading = f"### {clean_heading}"
        else:
            heading = f"#### {clean_heading}"

        ########################################################
        # MARKDOWN ASSEMBLY (FIXED: NO DICT IN STRING)
        ########################################################

        markdown_block = f"{heading}\n\n{section_text}"

        generated_sections.append({
            "heading": heading,
            "section_name": clean_heading,
            "content": section_text,
            "markdown": markdown_block,
            "sources": section_sources
        })

        ########################################################
        # MEMORY / SUMMARY BUFFER
        ########################################################

        previous_sections_summary += f"""
Section: {clean_heading}
Summary:
{section_text[:300]}
"""

        if len(previous_sections_summary) > 6000:
            previous_sections_summary = previous_sections_summary[-6000:]

    ########################################################
    # FINAL OUTPUT
    ########################################################

    return {
        "procedure_text": "# General Procedure\n\n" + "\n\n".join(
            s["markdown"] for s in generated_sections
        ),
        "sections": generated_sections
    }

## Step 5: Validates the generated text

### Step 5.1 Checks if requirements where included

In [16]:
# optimized
def validate_requirement_coverage(
    procedure_text,
    requirements,
    llm_client
):
    """
    Structured requirement coverage validation for SECTION-RAG SOPs.
    """

    ########################################################
    # PREPARE STRUCTURED REQUIREMENTS (NO TEXT FORMATTING)
    ########################################################

    structured_requirements = [
        {
            "id": i,
            "requirement_statement": r.get("requirement_statement"),
            "source_section_id": r.get("source_section_id"),
            "source_title": r.get("source_title"),
        }
        for i, r in enumerate(requirements)
    ]

    requirement_json = json.dumps(structured_requirements, indent=2)

    ########################################################
    # PROMPT
    ########################################################

    prompt = f"""
You are a strict validation engine for an industrial General Procedure (SOP).

SYSTEM DESIGN:
- SOP is generated using full-section retrieval (SECTION-RAG)
- Requirements are traceability anchors only

############################################################
REQUIREMENTS (STRUCTURED JSON - DO NOT MODIFY)
############################################################
{requirement_json}

############################################################
GENERATED PROCEDURE
############################################################
{procedure_text}

############################################################
TASK
############################################################

For EACH requirement:
- Check if it is explicitly covered in the procedure text

Classify as:
- Included
- Partially Included
- Missing

############################################################
RULES
############################################################

- Only use provided procedure text
- Do NOT assume missing content exists
- Do NOT hallucinate compliance
- Be strict and consistent
- Evaluate each requirement independently

############################################################
OUTPUT FORMAT (STRICT JSON ONLY)
############################################################

[
  {{
    "id": 0,
    "requirement_statement": "...",
    "status": "Included | Partially Included | Missing",
    "reason": "short explanation",
    "source_section_id": "...",
    "source_title": "..."
  }}
]
"""

    return llm_client.generate(prompt)

### Step 5.2 Peforms overall validation

In [17]:
def validate_procedure(
    procedure_text,
    requirements,
    llm_client
):
    """
    Enterprise-level SOP structural and consistency validation.
    """

    ########################################################
    # OPTIONAL: LIGHTWEIGHT REQUIREMENT SUMMARY (NOT CORE TRUTH)
    ########################################################
    # FIX: support both old and new pipeline format
    if isinstance(requirements, dict):
        requirements = (
            requirements.get("global_requirements", [])
            + requirements.get("retrieved_requirements", [])
        )

    # safety: filter bad entries
    requirements = [
        r for r in requirements
        if isinstance(r, dict)
    ]
    requirement_summary = [
        {
            "statement": r.get("requirement_statement"),
            "source": r.get("source_section_id")
        }
        for r in requirements
    ]

    ########################################################
    # PROMPT
    ########################################################

    prompt = f"""
You are a senior industrial General Procedure auditor.

This procedure has already passed REQUIREMENT COVERAGE VALIDATION.

Your role is NOW ONLY to evaluate:

- structural correctness
- procedural logic
- clarity and executability
- internal consistency

############################################################
INPUT: REQUIREMENT CONTEXT (REFERENCE ONLY)
############################################################
{json.dumps(requirement_summary, indent=2)}

############################################################
INPUT: GENERATED PROCEDURE
############################################################
{procedure_text}

############################################################
IMPORTANT SYSTEM CONTEXT
############################################################

- Requirement coverage has already been validated
- Do NOT re-check requirement completeness
- Focus only on SOP quality, structure, and correctness
- Treat requirements as contextual hints, not validation targets

############################################################
VALIDATION TASKS
############################################################

1. Structural Integrity
   - Missing SOP sections
   - Broken hierarchy or ordering

2. Procedural Flow Consistency
   - startup → operation → shutdown → maintenance order correctness

3. Unsupported or Illogical Statements
   - instructions that are not operationally executable
   - contradictions within procedure

4. Ambiguity / Non-Executable Language
   - vague terms
   - missing thresholds or measurable criteria

5. Cross-Section Consistency
   - conflicting instructions between sections

6. Overall Risk Assessment
   - Low / Medium / High based on operational reliability

############################################################
OUTPUT FORMAT (STRICT JSON ONLY)
############################################################

{{
  "structural_issues": [
    {{
      "issue": "",
      "missing_section": ""
    }}
  ],
  "flow_issues": [
    {{
      "issue": "",
      "location": ""
    }}
  ],
  "ambiguities": [
    {{
      "statement": "",
      "issue": ""
    }}
  ],
  "logical_conflicts": [
    {{
      "issue": "",
      "conflict": ""
    }}
  ],
  "unsupported_statements": [
    {{
      "statement": "",
      "reason": ""
    }}
  ],
  "overall_assessment": {{
    "rating": "Low | Medium | High",
    "summary": ""
  }}
}}

Return ONLY JSON.
No explanations.
No extra text.
"""
    
    return llm_client.generate(prompt)

## Step 6: Save doc file

### Without word sources

In [18]:
def save_procedure_docx(
    procedure,
    validation_text,
    generation_sources,
    output_path="output/generated_procedure.docx",
    output_path2="output/validation.docx"
):
    """
    Deterministic SOP + validation DOCX renderer (supports structured + legacy output).
    """

    ########################################################
    # PROCEDURE DOCUMENT
    ########################################################

    doc = Document()
    doc.add_heading("General Procedure", level=1)

    ########################################################
    # CASE 1 — NEW STRUCTURED FORMAT
    ########################################################

    if isinstance(procedure, dict) and "sections" in procedure:

        for section in procedure["sections"]:

            heading = section.get("heading", "Unknown Section")
            content = section.get("content", "")
            sources = section.get("sources", [])

            # SECTION HEADER
            doc.add_heading(heading.replace("##", "").strip(), level=2)

            # MAIN CONTENT
            doc.add_paragraph(content)

           

    ########################################################
    # CASE 2 — LEGACY STRING FORMAT (FALLBACK)
    ########################################################

    else:

        procedure_text = (
            procedure["procedure_text"]
            if isinstance(procedure, dict)
            else procedure
        )

        lines = procedure_text.split("\n")

        for line in lines:
            stripped = line.strip()

            if not stripped:
                continue

            if stripped.startswith("#### "):
                doc.add_heading(stripped[5:], level=4)

            elif stripped.startswith("### "):
                doc.add_heading(stripped[4:], level=3)

            elif stripped.startswith("## "):
                doc.add_heading(stripped[3:], level=2)

            elif stripped.startswith("# "):
                doc.add_heading(stripped[2:], level=1)

            elif stripped.startswith("- ") or stripped.startswith("* "):
                doc.add_paragraph(stripped[2:], style="List Bullet")

            else:
                doc.add_paragraph(stripped)


    doc.add_page_break()
    doc.add_heading("Sources Used (Deduplicated)", level=2)
    
    seen = set()
    
    for src in all_generation_sources:
        key = (src["document"], src["section_id"], src["title"])
        if key in seen:
            continue
        seen.add(key)
    
        doc.add_paragraph(
            f"- {src.get('document')} | "
            f"{src.get('section_id')} | "
            f"{src.get('title')}",
            style="Intense Quote"
        )
    doc.save(output_path)

    ########################################################
    # VALIDATION DOCUMENT (UNCHANGED)
    ########################################################

    doc2 = Document()
    doc2.add_heading("Validation Results", level=1)

    try:
        parsed = json.loads(validation_text)
        pretty = json.dumps(parsed, indent=2)

        for line in pretty.split("\n"):
            doc2.add_paragraph(line)

    except Exception:
        for line in validation_text.splitlines():
            doc2.add_paragraph(line)

    doc2.save(output_path2)

    return output_path, output_path2

### Word hyperlink version

In [19]:
def save_procedure_docx(
    procedure,
    validation_text,
    document_folder,
    output_path="output/generated_procedure.docx",
    output_path2="output/validation.docx"
):
    """
    Deterministic SOP + validation DOCX renderer
    (supports structured + legacy output).
    """

    import os
    import json

    from docx import Document
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn

    ########################################################
    # HYPERLINK HELPER
    ########################################################

    def add_hyperlink(paragraph, text, url):

        part = paragraph.part

        r_id = part.relate_to(
            url,
            reltype="http://schemas.openxmlformats.org/officeDocument/2006/relationships/hyperlink",
            is_external=True
        )

        hyperlink = OxmlElement("w:hyperlink")
        hyperlink.set(qn("r:id"), r_id)

        new_run = OxmlElement("w:r")
        rPr = OxmlElement("w:rPr")

        c = OxmlElement("w:color")
        c.set(qn("w:val"), "0000FF")
        rPr.append(c)

        u = OxmlElement("w:u")
        u.set(qn("w:val"), "single")
        rPr.append(u)

        new_run.append(rPr)

        text_el = OxmlElement("w:t")
        text_el.text = text
        new_run.append(text_el)

        hyperlink.append(new_run)
        paragraph._p.append(hyperlink)

        return hyperlink

    ########################################################
    # PROCEDURE DOCUMENT
    ########################################################

    doc = Document()
    doc.add_heading("General Procedure", level=1)

    ########################################################
    # CASE 1 — NEW STRUCTURED FORMAT
    ########################################################

    if isinstance(procedure, dict) and "sections" in procedure:

        ####################################################
        # RENDER PROCEDURE SECTIONS
        ####################################################

        for section in procedure["sections"]:

            heading = section.get("heading", "Unknown Section")
            content = section.get("content", "")

            # if content is structured dict
            if isinstance(content, dict):
                content = content.get("content", "")

            # SECTION HEADER
            clean_heading = (
                heading
                .replace("####", "")
                .replace("###", "")
                .replace("##", "")
                .strip()
            )

            doc.add_heading(clean_heading, level=2)

            # MAIN CONTENT
            doc.add_paragraph(content)

        ####################################################
        # GLOBAL SOURCE APPENDIX (DEDUPLICATED)
        ####################################################

        doc.add_page_break()
        doc.add_heading("Sources Used", level=2)

        seen = set()

        for section in procedure["sections"]:

            for src in section.get("sources", []):

                key = (
                    src.get("document"),
                    src.get("section_id"),
                    src.get("title")
                )

                if key in seen:
                    continue

                seen.add(key)

                ################################################
                # FILE URL
                ################################################

                file_path = os.path.abspath(
                    os.path.join(
                        document_folder,
                        src.get("document", "")
                    )
                )

                file_url = (
                    f"file:///{file_path.replace(os.sep, '/')}"
                )

                ################################################
                # DISPLAY TEXT
                ################################################

                display_text = (
                    f"{src.get('document')} | "
                    f"{src.get('section_id')} | "
                    f"{src.get('title')}"
                )

                ################################################
                # CLICKABLE LINK
                ################################################

                p = doc.add_paragraph(style="Intense Quote")

                add_hyperlink(
                    p,
                    display_text,
                    file_url
                )

    ########################################################
    # CASE 2 — LEGACY STRING FORMAT (FALLBACK)
    ########################################################

    else:

        procedure_text = (
            procedure["procedure_text"]
            if isinstance(procedure, dict)
            else procedure
        )

        lines = procedure_text.split("\n")

        for line in lines:

            stripped = line.strip()

            if not stripped:
                continue

            ####################################################
            # HEADINGS
            ####################################################

            if stripped.startswith("#### "):
                doc.add_heading(stripped[5:], level=4)

            elif stripped.startswith("### "):
                doc.add_heading(stripped[4:], level=3)

            elif stripped.startswith("## "):
                doc.add_heading(stripped[3:], level=2)

            elif stripped.startswith("# "):
                doc.add_heading(stripped[2:], level=1)

            ####################################################
            # BULLETS
            ####################################################

            elif stripped.startswith("- ") or stripped.startswith("* "):
                doc.add_paragraph(
                    stripped[2:],
                    style="List Bullet"
                )

            ####################################################
            # NORMAL TEXT
            ####################################################

            else:
                doc.add_paragraph(stripped)

    ########################################################
    # SAVE PROCEDURE DOC
    ########################################################

    doc.save(output_path)

    ########################################################
    # VALIDATION DOCUMENT
    ########################################################

    doc2 = Document()
    doc2.add_heading("Validation Results", level=1)

    try:

        parsed = json.loads(validation_text)

        pretty = json.dumps(
            parsed,
            indent=2
        )

        for line in pretty.split("\n"):
            doc2.add_paragraph(line)

    except Exception:

        for line in validation_text.splitlines():
            doc2.add_paragraph(line)

    ########################################################
    # SAVE VALIDATION DOC
    ########################################################

    doc2.save(output_path2)

    return output_path, output_path2

In [20]:
def add_hyperlink(paragraph, text, url):
    part = paragraph.part
    r_id = part.relate_to(
        url,
        reltype="http://schemas.openxmlformats.org/officeDocument/2006/relationships/hyperlink",
        is_external=True
    )

    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), r_id)

    new_run = OxmlElement("w:r")
    rPr = OxmlElement("w:rPr")

    c = OxmlElement("w:color")
    c.set(qn("w:val"), "0000FF")
    rPr.append(c)

    u = OxmlElement("w:u")
    u.set(qn("w:val"), "single")
    rPr.append(u)

    new_run.append(rPr)

    text_el = OxmlElement("w:t")
    text_el.text = text
    new_run.append(text_el)

    hyperlink.append(new_run)
    paragraph._p.append(hyperlink)

    return hyperlink

## Run the workflow

In [21]:
# optimzed
# Optimized
def run_workflow(
    user_request,
    document_folder,
    llm_client,
    procedure_template,
    progress_callback=None
):
    """
    End-to-end SECTION-RAG SOP generation workflow (clean architecture).
    """

    import time

    start_time = time.time()
    step_timings = {}
    step_start = None
    last_step = None

    ########################################################
    # PROGRESS + TIMING HELPERS
    ########################################################

    def update(step, percent):
        if progress_callback:
            progress_callback(step, percent)
        else:
            print(step)

    def mark_step(name):
        nonlocal step_start, last_step

        if step_start is not None and last_step is not None:
            step_timings[last_step] = round(time.time() - step_start, 2)

        step_start = time.time()
        last_step = name

    ########################################################
    # STEP 1 — LOAD SECTIONS
    ########################################################

    mark_step("Loading documents")
    update("STEP 1 — Loading documents", 5)

    sections = load_all_documents(document_folder)

    update(f"Loaded sections: {len(sections)}", 10)

    ########################################################
    # PROVENANCE (SECTION TRUTH LAYER)
    ########################################################

    retrieval_sources = [
        {
            "document": s.get("document", "unknown"),
            "section_id": s.get("section_id"),
            "title": s.get("title"),
            "level": s.get("level"),
            "retrieval_text": s.get("retrieval_text", "")
        }
        for s in sections
    ]

    ########################################################
    # STEP 2 — BUILD SECTION RETRIEVAL INDEX
    ########################################################

    mark_step("Building retrieval system")
    update("STEP 2 — Building retrieval system", 20)

    retrieval_system = build_section_index(sections)

    ########################################################
    # STEP 3 — GLOBAL REQUIREMENT EXTRACTION (FULL CORPUS)
    ########################################################

    mark_step("Extracting global requirements")
    update("STEP 3 — Extracting global requirements", 30)

    requirements = build_requirement_database(sections, llm_client)

    update(f"Global requirements: {len(requirements)}", 40)

    ########################################################
    # STEP 4 — DEDUPLICATION (GLOBAL)
    ########################################################

    mark_step("Deduplicating requirements")
    update("STEP 4 — Deduplicating requirements", 50)

    requirements = deduplicate_requirements(requirements)

    update(f"Unique global requirements: {len(requirements)}", 60)

    ########################################################
    # STEP 5 — RETRIEVE RELEVANT SECTIONS (GENERATION CONTEXT)
    ########################################################

    mark_step("Retrieving relevant sections")
    update("STEP 5 — Retrieving sections", 65)

    relevant_sections = retrieve_sections(
        query=user_request,
        retrieval_system=retrieval_system,
        k=25
    )

    ########################################################
    # STEP 6 — LOCAL REQUIREMENT EXTRACTION (RAG-ALIGNED)
    ########################################################

    mark_step("Extracting retrieved requirements")
    update("STEP 6 — Extracting retrieved requirements", 68)

    retrieved_requirements = build_requirement_database(
        relevant_sections,
        llm_client
    )

    update(f"Retrieved requirements: {len(retrieved_requirements)}", 69)

    ########################################################
    # STEP 7 — PROCEDURE GENERATION
    ########################################################

    mark_step("Generating procedure")
    update("STEP 7 — Generating procedure", 70)


    procedure_obj = generate_general_procedure(
        user_request=user_request,
        retrieval_system=retrieval_system,
        procedure_template=procedure_template,
        llm_client=llm_client
        )
        
    procedure = procedure_obj["procedure_text"]
    procedure_sections = procedure_obj["sections"]
    
    all_generation_sources = []
    seen = set()
    
    for s in procedure_sections:
        for src in s.get("sources", []):
            
            key = (
                src.get("document"),
                src.get("section_id"),
                src.get("title")
            )
    
            if key in seen:
                continue
    
            seen.add(key)
            all_generation_sources.append(src)
        
    ########################################################
    # STEP 8 — REQUIREMENT COVERAGE VALIDATION (LOCAL ALIGNMENT)
    ########################################################

    mark_step("Requirement coverage validation")
    update("STEP 8 — Requirement coverage validation", 82)

    coverage = validate_requirement_coverage(
        procedure,
        retrieved_requirements,
        llm_client
    )

    ########################################################
    # STEP 9 — ENTERPRISE VALIDATION (GLOBAL + LOCAL CONTEXT)
    ########################################################

    mark_step("Enterprise validation")
    update("STEP 9 — Enterprise validation", 90)

    validation = validate_procedure(
        procedure,
        {
            "global_requirements": requirements,
            "retrieved_requirements": retrieved_requirements
        },
        llm_client
    )

    ########################################################
    # STEP 10 — SAVE OUTPUTS
    ########################################################

    mark_step("Saving document")
    update("STEP 10 — Saving document", 97)

    output_path, output_path2 = save_procedure_docx(
        procedure_obj,
        validation,
        document_folder
        #all_generation_sources
    )

    ########################################################
    # FINALIZE TIMING
    ########################################################

    if step_start is not None and last_step is not None:
        step_timings[last_step] = round(time.time() - step_start, 2)

    total_time = round(time.time() - start_time, 2)

    update(f"DONE — Completed in {total_time}s", 100)

    ########################################################
    # RETURN RESULTS
    ########################################################

    return {
        "procedure": procedure,
        "coverage": coverage,
        "validation": validation,
        "output_path": output_path,
        "output_path_validation": output_path2,

        # core artifacts
        "requirements": requirements,
        "retrieved_requirements": retrieved_requirements,
        "procedure_template": procedure_template,

        # analytics
        "timings": step_timings,
        "runtime_seconds": total_time,

        # transparency layer
        "retrieval_sources": retrieval_sources,
        "documents_loaded": len(sections),

        # stats
        "total_requirements": len(requirements),
        "total_retrieved_requirements": len(retrieved_requirements),
        #generation sources
        "generation_sources": all_generation_sources
    }

In [22]:
# Optimized
def run_workflow(
    user_request,
    document_folder,
    llm_client,
    procedure_template,
    progress_callback=None
):
    """
    End-to-end SECTION-RAG SOP generation workflow.

    Correct order:
    1. Load + parse documents
    2. Build retrieval index
    3. Retrieve relevant sections (RAG FIRST)
    4. Extract requirements ONLY from retrieved context
    5. Generate procedure
    6. Validate
    7. Save outputs
    """

    import time

    start_time = time.time()
    step_timings = {}
    step_start = None
    last_step = None

    ########################################################
    # PROGRESS + TIMING HELPERS
    ########################################################

    def update(step, percent):
        if progress_callback:
            progress_callback(step, percent)
        else:
            print(step)

    def mark_step(name):
        nonlocal step_start, last_step

        if step_start is not None and last_step is not None:
            step_timings[last_step] = round(
                time.time() - step_start,
                2
            )

        step_start = time.time()
        last_step = name

    ########################################################
    # STEP 1 — LOAD DOCUMENTS
    ########################################################

    mark_step("Loading documents")
    update("STEP 1 — Loading documents", 5)

    sections = load_all_documents(document_folder)

    update(f"Loaded sections: {len(sections)}", 10)

    ########################################################
    # TRANSPARENCY LAYER
    ########################################################

    retrieval_sources = [
        {
            "document": s.get("document", "unknown"),
            "section_id": s.get("section_id"),
            "title": s.get("title"),
            "level": s.get("level"),
            "retrieval_text": s.get("retrieval_text", "")
        }
        for s in sections
    ]

    ########################################################
    # STEP 2 — BUILD RETRIEVAL SYSTEM
    ########################################################

    mark_step("Building retrieval system")
    update("STEP 2 — Building retrieval system", 20)

    retrieval_system = build_section_index(sections)

    ########################################################
    # STEP 3 — RETRIEVE RELEVANT SECTIONS (RAG FIRST)
    ########################################################

    mark_step("Retrieving relevant sections")
    update("STEP 3 — Retrieving sections", 35)

    relevant_sections = retrieve_sections(
        query=user_request,
        retrieval_system=retrieval_system,
        k=25
    )

    update(
        f"Retrieved sections: {len(relevant_sections)}",
        45
    )

    ########################################################
    # STEP 4 — EXTRACT REQUIREMENTS ONLY FROM
    # RETRIEVED SECTIONS
    ########################################################

    mark_step("Extracting requirements")
    update("STEP 4 — Extracting requirements", 55)

    retrieved_requirements = build_requirement_database(
        relevant_sections,
        llm_client
    )

    ########################################################
    # DEDUPLICATE REQUIREMENTS
    ########################################################

    retrieved_requirements = deduplicate_requirements(
        retrieved_requirements
    )

    update(
        f"Unique retrieved requirements: "
        f"{len(retrieved_requirements)}",
        65
    )

    ########################################################
    # STEP 5 — GENERATE PROCEDURE
    ########################################################

    mark_step("Generating procedure")
    update("STEP 5 — Generating procedure", 72)

    procedure_obj = generate_general_procedure(
        user_request=user_request,
        retrieval_system=retrieval_system,
        procedure_template=procedure_template,
        llm_client=llm_client
    )

    procedure = procedure_obj["procedure_text"]
    procedure_sections = procedure_obj["sections"]

    ########################################################
    # DEDUPLICATED GENERATION SOURCES
    ########################################################

    all_generation_sources = []
    seen = set()

    for section in procedure_sections:

        for src in section.get("sources", []):

            key = (
                src.get("document"),
                src.get("section_id"),
                src.get("title")
            )

            if key in seen:
                continue

            seen.add(key)
            all_generation_sources.append(src)

    ########################################################
    # STEP 6 — COVERAGE VALIDATION
    ########################################################

    mark_step("Requirement coverage validation")
    update("STEP 6 — Coverage validation", 82)

    coverage = validate_requirement_coverage(
        procedure,
        retrieved_requirements,
        llm_client
    )

    ########################################################
    # STEP 7 — PROCEDURE VALIDATION
    ########################################################

    mark_step("Enterprise validation")
    update("STEP 7 — Enterprise validation", 90)

    validation = validate_procedure(
        procedure,
        {
            "retrieved_requirements": retrieved_requirements
        },
        llm_client
    )

    ########################################################
    # STEP 8 — SAVE OUTPUTS
    ########################################################

    mark_step("Saving documents")
    update("STEP 8 — Saving documents", 97)

    output_path, output_path2 = save_procedure_docx(
        procedure=procedure_obj,
        validation_text=validation,
        document_folder=document_folder
    )

    ########################################################
    # FINALIZE TIMING
    ########################################################

    if step_start is not None and last_step is not None:
        step_timings[last_step] = round(
            time.time() - step_start,
            2
        )

    total_time = round(time.time() - start_time, 2)

    update(
        f"DONE — Completed in {total_time}s",
        100
    )

    ########################################################
    # RETURN RESULTS
    ########################################################

    return {
        "procedure": procedure,
        "coverage": coverage,
        "validation": validation,

        "output_path": output_path,
        "output_path_validation": output_path2,

        # retrieved RAG context
        "relevant_sections": relevant_sections,

        # requirements
        "retrieved_requirements": retrieved_requirements,

        # provenance
        "retrieval_sources": retrieval_sources,
        "generation_sources": all_generation_sources,

        # metadata
        "documents_loaded": len(sections),
        "total_retrieved_requirements": len(
            retrieved_requirements
        ),

        # analytics
        "timings": step_timings,
        "runtime_seconds": total_time,

        # template
        "procedure_template": procedure_template
    }

In [23]:
############################################################
# EXAMPLE EXECUTION
############################################################
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

if __name__ == "__main__":

    result = run_workflow(
        user_request="Tell me how to peform a rubber tire change for the H45-65XM MODELS",
        document_folder=r"knowledge_bases/pressure_system",
        llm_client=llm_client,
        procedure_template=procedure_template,
        progress_callback=None
    )

    #print(result["procedure"])


F:\Pytorch\venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
W0514 20:22:56.486000 3196 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
F:\Pytorch\venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


STEP 1 — Loading documents
Loaded sections: 163
STEP 2 — Building retrieval system
STEP 3 — Retrieving sections
Retrieved sections: 25
STEP 4 — Extracting requirements
Unique retrieved requirements: 84
STEP 5 — Generating procedure
STEP 6 — Coverage validation
STEP 7 — Enterprise validation
STEP 8 — Saving documents
DONE — Completed in 282.89s


In [24]:
for row in result["generation_sources"][:]:
    print("=" * 50)
    print(f"Document     : {row['document']}")
    print(f"Section ID   : {row['section_id']}: {row['title']}")
    print()

Document     : maintainence.pdf.docx
Section ID   : None: SOLID RUBBER TIRE, CHANGE H45-65XM MODELS

Document     : maintainence.pdf.docx
Section ID   : None: SOLID RUBBER TIRE, CHANGE S40-65XM MODELS

Document     : maintainence.pdf.docx
Section ID   : None: Remove Wheels From Lift Truck

Document     : maintainence.pdf.docx
Section ID   : None: WARNING

Document     : maintainence.pdf.docx
Section ID   : None: Remove and Install Tire on Wheel

Document     : maintainence.pdf.docx
Section ID   : None: ADD AIR TO PNEUMATIC TIRES

Document     : maintainence.pdf.docx
Section ID   : None: CAUTION

Document     : maintainence.pdf.docx
Section ID   : None: Other ManualsLib Projects



In [25]:
result["timings"]

{'Loading documents': 0.88,
 'Building retrieval system': 0.52,
 'Retrieving relevant sections': 0.05,
 'Extracting requirements': 159.33,
 'Generating procedure': 80.11,
 'Requirement coverage validation': 20.13,
 'Enterprise validation': 21.79,
 'Saving documents': 0.08}

In [ ]:
### Pipeline
# Enterprise Procedural RAG Pipeline — Requirement-Driven Architecture

############################################################
# IMPORTS
############################################################

import os
import re
import json
import numpy as np
import pandas as pd
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
import ollama
import time
from llm_client import LLMClient

############################################################
# EMBEDDING MODEL
############################################################

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def detect_procedure_type(user_request):
    """
    Detects the operational procedure type from a user request.

    This function performs lightweight keyword-based classification
    to identify the intended industrial procedure category.

    Supported procedure types:
    - startup
    - shutdown
    - maintenance
    - inspection
    - general (fallback)

    Args:
        user_request (str):
            User query or procedure generation request.

    Returns:
        str:
            Detected procedure type.
    """

    text = user_request.lower()

    if "startup" in text:
        return "startup"
    if "shutdown" in text:
        return "shutdown"
    if "maintenance" in text:
        return "maintenance"
    if "inspection" in text:
        return "inspection"

    return "general"

HEADINGS = {
    "rubrik 1",
    "rubrik 2",
    "rubrik 3",
    "rubrik 4",
    "heading 1",
    "heading 2",
    "heading 3",
    "heading 4",
}

SECTION_RE = re.compile(r"^([\d.]+)\s+(.*)")
SPACE_RE = re.compile(r"\s+")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def parse_word_document(filepath):
    doc = Document(filepath)
    filename = os.path.basename(filepath)

    sections = []
    current = None
    buffer = []

    paragraphs = [
        (p.text.strip(), SPACE_RE.sub(" ", p.style.name.lower()))
        for p in doc.paragraphs
        if p.text.strip()
    ]

    def flush():
        nonlocal buffer, current

        if not current:
            return

        raw = "\n".join(buffer)

        header = f"{current['section_id'] or ''} {current['title']}".strip()

        sections.append({
            "document": filename,
            **current,
            "raw_text": raw,
            "sentences": buffer,
            "section_text": f"{header}\n{raw}",
            "retrieval_text": f"{header} {raw}"
        })

        buffer = []

    for text, style in paragraphs:

        # 🔥 FIX 1: normalize style more aggressively
        style = style.strip().lower()

        if style in HEADINGS:

            flush()

            match = SECTION_RE.match(text)

            if match:
                sid, title = match.groups()
                level = sid.count(".") + 1
            else:
                sid, title, level = None, text, 1

            current = {
                "section_id": sid,
                "title": title,
                "level": level
            }

        else:
            buffer.append(text)

    flush()

    if sections:
        return sections

    fallback = " ".join(text for text, _ in paragraphs)
    fallback = " ".join(fallback.split()[:200])

    return [{
        "document": filename,
        "section_id": "0",
        "title": "Full Document (Auto Chunk)",
        "level": 0,
        "raw_text": fallback,
        "sentences": SENTENCE_RE.split(fallback),
        "section_text": f"0 Full Document (Auto Chunk)\n{fallback}",
        "retrieval_text": f"0 Full Document (Auto Chunk) {fallback}"
    }]
def load_all_documents(folder_path):
    """
    Loads and parses all .docx documents from a folder.

    This function:
    - Iterates through every .docx file in the provided folder
    - Parses each document using parse_word_document()
    - Aggregates all extracted sections into a single list

    Args:
        folder_path (str):
            Path to the folder containing Word documents.

    Returns:
        list[dict]:
            Combined list of parsed sections from all documents.
    """
    all_sections = []

    for file in os.listdir(folder_path):

        if file.endswith(".docx"):

            path = os.path.join(folder_path, file)
            sections = parse_word_document(path)

            all_sections.extend(sections)

    return all_sections

def build_section_index(sections):
    """
    Builds a hybrid section-level retrieval index.

    Retrieval units are full document sections.

    The index combines:
    - semantic embeddings
    - TF-IDF lexical retrieval
    - section metadata

    Args:
        sections (list[dict]):
            Parsed document sections.

    Returns:
        dict:
            Hybrid retrieval system.
    """

    if not sections:
        raise ValueError("No sections found")

    ########################################################
    # RETRIEVAL TEXTS
    ########################################################

    section_texts = [
        s["retrieval_text"]
        for s in sections
    ]

    ########################################################
    # SEMANTIC EMBEDDINGS
    ########################################################

    section_embeddings = embedding_model.encode(
        section_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    ########################################################
    # LEXICAL INDEX
    ########################################################

    tfidf = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    tfidf_matrix = tfidf.fit_transform(section_texts)

    ########################################################
    # RETURN HYBRID INDEX
    ########################################################

    return {
        "sections": sections,
        "section_texts": section_texts,
        "section_embeddings": np.asarray(section_embeddings),
        "tfidf": tfidf,
        "tfidf_matrix": tfidf_matrix
    }
def retrieve_sections(
    query,
    retrieval_system,
    section_heading=None,
    k=10,
    alpha=0.7
):
    """
    Hybrid section retrieval using:
    - semantic embeddings
    - TF-IDF lexical matching
    """

    ########################################################
    # QUERY CONSTRUCTION
    ########################################################

    if section_heading:
        query = f"{query}\n{section_heading}"

    ########################################################
    # RETRIEVAL COMPONENTS
    ########################################################

    sections = retrieval_system["sections"]

    section_embeddings = retrieval_system["section_embeddings"]

    ########################################################
    # SEMANTIC SEARCH
    ########################################################

    query_emb = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    semantic_scores = section_embeddings @ query_emb

    ########################################################
    # LEXICAL SEARCH
    ########################################################

    tfidf_vec = retrieval_system["tfidf"].transform([query])

    lexical_scores = (
        tfidf_vec @ retrieval_system["tfidf_matrix"].T
    ).toarray()[0]

    ########################################################
    # SCORE NORMALIZATION
    ########################################################

    def normalize(x):
        rng = np.ptp(x)
        return np.zeros_like(x) if rng == 0 else (x - x.min()) / rng

    semantic_scores = normalize(semantic_scores)
    lexical_scores = normalize(lexical_scores)

    ########################################################
    # HYBRID SCORING
    ########################################################

    scores = (
        alpha * semantic_scores
        + (1 - alpha) * lexical_scores
    )

    ########################################################
    # TOP-K (WITHOUT FULL SORT)
    ########################################################
    if len(scores) == 0:
        return []

    k = min(k, len(scores))
    top_idx = np.argpartition(scores, -k)[-k:]

    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    ########################################################
    # RETURN RANKED SECTIONS
    ########################################################

    return [sections[i] for i in top_idx]

JSON_RE = re.compile(r"\[.*?\]", re.DOTALL)
CODEBLOCK_RE = re.compile(r"```(?:json)?|```", re.I)

REQUIRED_FIELDS = {
    "requirement_type",
    "operational_phase",
    "requirement_statement",
    "criticality"
}
def extract_requirements(section, llm_client):
    """
    Extract structured requirements from a single section.
    """

    prompt = (
        "You are an industrial operations analyst.\n\n"
        "Extract ONLY explicit operational requirements.\n\n"
        f"SECTION:\n{section['section_id']} {section['title']}\n\n"
        f"CONTENT:\n{section['raw_text']}\n\n"
        "Return JSON ONLY as a list of objects:\n"
        "[{\n"
        "  \"requirement_type\": \"...\",\n"
        "  \"operational_phase\": \"...\",\n"
        "  \"requirement_statement\": \"...\",\n"
        "  \"criticality\": \"...\"\n"
        "}]\n\n"
        "Rules:\n"
        "- No hallucinations\n"
        "- No summarization\n"
        "- Only explicit requirements\n"
    )

    content = llm_client.generate(prompt)

    ########################################################
    # CLEAN OUTPUT
    ########################################################

    content = CODEBLOCK_RE.sub("", content)

    match = JSON_RE.search(content)

    if not match:
        return []

    try:
        reqs = json.loads(match.group(0))
    except json.JSONDecodeError:
        return []

    ########################################################
    # VALIDATION + ENRICHMENT
    ########################################################

    base_meta = {
        "source_document": section.get("document"),
        "source_section_id": section.get("section_id"),
        "source_title": section.get("title")
    }

    cleaned = []

    for r in reqs:
        if not isinstance(r, dict):
            continue

        if not REQUIRED_FIELDS.issubset(r.keys()):
            continue

        cleaned.append({**r, **base_meta})

    return cleaned
def build_requirement_database(sections, llm_client):
    """
    Extract requirements ONLY from retrieved sections.
    """

    return [
        req
        for section in sections
        if section.get("raw_text")
        for req in extract_requirements(section, llm_client)
    ]
def deduplicate_requirements(requirements, threshold=0.9):
    if not requirements:
        return []

    texts = [r["requirement_statement"] for r in requirements]

    emb = embedding_model.encode(texts, normalize_embeddings=True)

    kept = []

    for i, e in enumerate(emb):
        if all((e @ emb[j]) < threshold for j in kept):
            kept.append(i)

    return [requirements[i] for i in kept]
def format_requirements(requirements):
    """
    Formats requirements into LLM-grounded context block.
    """

    blocks = []

    for i, r in enumerate(requirements, 1):

        blocks.append(
f"""--- REQUIREMENT {i} ---
Type: {r.get('requirement_type', 'unknown')}
Phase: {r.get('operational_phase', 'unknown')}
Criticality: {r.get('criticality', 'unknown')}
Statement: {r.get('requirement_statement', 'unknown')}
Source: {r.get('source_document', 'unknown')} | {r.get('source_section_id', 'UNMAPPED')} | {r.get('source_title', 'unknown')}
"""
        )

    return "\n".join(blocks)
# Optimized
def generate_section(section_name, sections, user_request, llm_client):
    """
    Generates a grounded General Procedure section using retrieved document sections.
    """

    ########################################################
    # GROUNDING CONTEXT (SECTION-BASED, NOT REQUIREMENTS)
    ########################################################

    section_text = "\n\n".join(
        f"""--- SECTION ---
Title: {s.get('title', 'unknown')}
ID: {s.get('section_id', 'unknown')}

Content:
{s.get('retrieval_text', s.get('raw_text', ''))}
"""
        for s in sections
    )

    ########################################################
    # PROVENANCE TRACKING (NEW)
    ########################################################

    source_sections = [
        {
            "document": s.get("document", "unknown"),
            "section_id": s.get("section_id"),
            "title": s.get("title"),
            "level": s.get("level")
        }
        for s in sections
    ]

    ########################################################
    # PROMPT
    ########################################################

    prompt = f"""You are generating a General Procedure (SOP) section.

Use ONLY the provided retrieved sections as grounding material.

User Request:
{user_request}

Target Section:
{section_name}

############################################################
RETRIEVED SECTIONS
############################################################
{section_text}

############################################################
RULES
############################################################
- Use ONLY retrieved information
- Do not introduce external knowledge
- Do not hallucinate missing steps
- Do not include metadata (IDs, source labels)
- Write only the General Procedure section content

############################################################
OUTPUT FORMAT
############################################################
- Clear procedural structure
- Logical step ordering
- Bullet points where appropriate

Return ONLY the section content.
"""

    ########################################################
    # OUTPUT (STRUCTURED)
    ########################################################

    return {
        "content": str(llm_client.generate(prompt)),
        "sources": source_sections
    }
def generate_general_procedure(
    user_request,
    retrieval_system,
    llm_client,
    procedure_template
):
    """
    Template-driven SECTION-RAG SOP generation with clean structured output.
    """

    import re

    procedure_type = detect_procedure_type(user_request)

    template_lines = [
        line.strip()
        for line in procedure_template.splitlines()
        if line.strip()
    ]

    generated_sections = []
    previous_sections_summary = ""

    for line in template_lines:

        clean_heading = re.sub(r"^\d+(\.\d+)*\s*", "", line).strip()

        query = f"{user_request} {clean_heading}"

        retrieved_sections = retrieve_sections(
            query=query,
            retrieval_system=retrieval_system,
            k=10
        )

        ########################################################
        # SECTION GENERATION (FIXED STRUCTURE USAGE)
        ########################################################

        section_obj = generate_section(
            section_name=clean_heading,
            sections=retrieved_sections,
            user_request=user_request,
            llm_client=llm_client
        )

        section_text = section_obj["content"]
        section_sources = section_obj["sources"]

        ########################################################
        # HEADING LEVEL LOGIC
        ########################################################

        level = line.count(".") + 1

        if level <= 1:
            heading = f"## {clean_heading}"
        elif level == 2:
            heading = f"### {clean_heading}"
        else:
            heading = f"#### {clean_heading}"

        ########################################################
        # MARKDOWN ASSEMBLY (FIXED: NO DICT IN STRING)
        ########################################################

        markdown_block = f"{heading}\n\n{section_text}"

        generated_sections.append({
            "heading": heading,
            "section_name": clean_heading,
            "content": section_text,
            "markdown": markdown_block,
            "sources": section_sources
        })

        ########################################################
        # MEMORY / SUMMARY BUFFER
        ########################################################

        previous_sections_summary += f"""
Section: {clean_heading}
Summary:
{section_text[:300]}
"""

        if len(previous_sections_summary) > 6000:
            previous_sections_summary = previous_sections_summary[-6000:]

    ########################################################
    # FINAL OUTPUT
    ########################################################

    return {
        "procedure_text": "# General Procedure\n\n" + "\n\n".join(
            s["markdown"] for s in generated_sections
        ),
        "sections": generated_sections
    }
# optimized
def validate_requirement_coverage(
    procedure_text,
    requirements,
    llm_client
):
    """
    Structured requirement coverage validation for SECTION-RAG SOPs.
    """

    ########################################################
    # PREPARE STRUCTURED REQUIREMENTS (NO TEXT FORMATTING)
    ########################################################

    structured_requirements = [
        {
            "id": i,
            "requirement_statement": r.get("requirement_statement"),
            "source_section_id": r.get("source_section_id"),
            "source_title": r.get("source_title"),
        }
        for i, r in enumerate(requirements)
    ]

    requirement_json = json.dumps(structured_requirements, indent=2)

    ########################################################
    # PROMPT
    ########################################################

    prompt = f"""
You are a strict validation engine for an industrial General Procedure (SOP).

SYSTEM DESIGN:
- SOP is generated using full-section retrieval (SECTION-RAG)
- Requirements are traceability anchors only

############################################################
REQUIREMENTS (STRUCTURED JSON - DO NOT MODIFY)
############################################################
{requirement_json}

############################################################
GENERATED PROCEDURE
############################################################
{procedure_text}

############################################################
TASK
############################################################

For EACH requirement:
- Check if it is explicitly covered in the procedure text

Classify as:
- Included
- Partially Included
- Missing

############################################################
RULES
############################################################

- Only use provided procedure text
- Do NOT assume missing content exists
- Do NOT hallucinate compliance
- Be strict and consistent
- Evaluate each requirement independently

############################################################
OUTPUT FORMAT (STRICT JSON ONLY)
############################################################

[
  {{
    "id": 0,
    "requirement_statement": "...",
    "status": "Included | Partially Included | Missing",
    "reason": "short explanation",
    "source_section_id": "...",
    "source_title": "..."
  }}
]
"""

    return llm_client.generate(prompt)
def validate_procedure(
    procedure_text,
    requirements,
    llm_client
):
    """
    Enterprise-level SOP structural and consistency validation.
    """

    ########################################################
    # OPTIONAL: LIGHTWEIGHT REQUIREMENT SUMMARY (NOT CORE TRUTH)
    ########################################################
    # FIX: support both old and new pipeline format
    if isinstance(requirements, dict):
        requirements = (
            requirements.get("global_requirements", [])
            + requirements.get("retrieved_requirements", [])
        )

    # safety: filter bad entries
    requirements = [
        r for r in requirements
        if isinstance(r, dict)
    ]
    requirement_summary = [
        {
            "statement": r.get("requirement_statement"),
            "source": r.get("source_section_id")
        }
        for r in requirements
    ]

    ########################################################
    # PROMPT
    ########################################################

    prompt = f"""
You are a senior industrial General Procedure auditor.

This procedure has already passed REQUIREMENT COVERAGE VALIDATION.

Your role is NOW ONLY to evaluate:

- structural correctness
- procedural logic
- clarity and executability
- internal consistency

############################################################
INPUT: REQUIREMENT CONTEXT (REFERENCE ONLY)
############################################################
{json.dumps(requirement_summary, indent=2)}

############################################################
INPUT: GENERATED PROCEDURE
############################################################
{procedure_text}

############################################################
IMPORTANT SYSTEM CONTEXT
############################################################

- Requirement coverage has already been validated
- Do NOT re-check requirement completeness
- Focus only on SOP quality, structure, and correctness
- Treat requirements as contextual hints, not validation targets

############################################################
VALIDATION TASKS
############################################################

1. Structural Integrity
   - Missing SOP sections
   - Broken hierarchy or ordering

2. Procedural Flow Consistency
   - startup → operation → shutdown → maintenance order correctness

3. Unsupported or Illogical Statements
   - instructions that are not operationally executable
   - contradictions within procedure

4. Ambiguity / Non-Executable Language
   - vague terms
   - missing thresholds or measurable criteria

5. Cross-Section Consistency
   - conflicting instructions between sections

6. Overall Risk Assessment
   - Low / Medium / High based on operational reliability

############################################################
OUTPUT FORMAT (STRICT JSON ONLY)
############################################################

{{
  "structural_issues": [
    {{
      "issue": "",
      "missing_section": ""
    }}
  ],
  "flow_issues": [
    {{
      "issue": "",
      "location": ""
    }}
  ],
  "ambiguities": [
    {{
      "statement": "",
      "issue": ""
    }}
  ],
  "logical_conflicts": [
    {{
      "issue": "",
      "conflict": ""
    }}
  ],
  "unsupported_statements": [
    {{
      "statement": "",
      "reason": ""
    }}
  ],
  "overall_assessment": {{
    "rating": "Low | Medium | High",
    "summary": ""
  }}
}}

Return ONLY JSON.
No explanations.
No extra text.
"""
    
    return llm_client.generate(prompt)

def save_procedure_docx(
    procedure,
    validation_text,
    all_generation_sources,
    output_path="output/generated_procedure.docx",
    output_path2="output/validation.docx"
):
    """
    Deterministic SOP + validation DOCX renderer (supports structured + legacy output).
    """

    ########################################################
    # PROCEDURE DOCUMENT
    ########################################################

    doc = Document()
    doc.add_heading("General Procedure", level=1)

    ########################################################
    # CASE 1 — NEW STRUCTURED FORMAT
    ########################################################

    if isinstance(procedure, dict) and "sections" in procedure:

        for section in procedure["sections"]:

            heading = section.get("heading", "Unknown Section")
            content = section.get("content", "")
            sources = section.get("sources", [])

            # SECTION HEADER
            doc.add_heading(heading.replace("##", "").strip(), level=2)

            # MAIN CONTENT
            doc.add_paragraph(content)

           

    ########################################################
    # CASE 2 — LEGACY STRING FORMAT (FALLBACK)
    ########################################################

    else:

        procedure_text = (
            procedure["procedure_text"]
            if isinstance(procedure, dict)
            else procedure
        )

        lines = procedure_text.split("\n")

        for line in lines:
            stripped = line.strip()

            if not stripped:
                continue

            if stripped.startswith("#### "):
                doc.add_heading(stripped[5:], level=4)

            elif stripped.startswith("### "):
                doc.add_heading(stripped[4:], level=3)

            elif stripped.startswith("## "):
                doc.add_heading(stripped[3:], level=2)

            elif stripped.startswith("# "):
                doc.add_heading(stripped[2:], level=1)

            elif stripped.startswith("- ") or stripped.startswith("* "):
                doc.add_paragraph(stripped[2:], style="List Bullet")

            else:
                doc.add_paragraph(stripped)


    doc.add_page_break()
    doc.add_heading("Sources Used (Deduplicated)", level=2)
    
    seen = set()
    
    for src in all_generation_sources:
        key = (src["document"], src["section_id"], src["title"])
        if key in seen:
            continue
        seen.add(key)
    
        doc.add_paragraph(
            f"- {src.get('document')} | "
            f"{src.get('section_id')} | "
            f"{src.get('title')}",
            style="Intense Quote"
        )
    doc.save(output_path)

    ########################################################
    # VALIDATION DOCUMENT (UNCHANGED)
    ########################################################

    doc2 = Document()
    doc2.add_heading("Validation Results", level=1)

    try:
        parsed = json.loads(validation_text)
        pretty = json.dumps(parsed, indent=2)

        for line in pretty.split("\n"):
            doc2.add_paragraph(line)

    except Exception:
        for line in validation_text.splitlines():
            doc2.add_paragraph(line)

    doc2.save(output_path2)

    return output_path, output_path2
# optimzed
def run_workflow(
    user_request,
    document_folder,
    llm_client,
    procedure_template,
    progress_callback=None
):
    """
    End-to-end SECTION-RAG SOP generation workflow.

    Correct order:
    1. Load + parse documents
    2. Build retrieval index
    3. Retrieve relevant sections (RAG FIRST)
    4. Extract requirements ONLY from retrieved context
    5. Generate procedure
    6. Validate
    7. Save outputs
    """

    import time

    start_time = time.time()
    step_timings = {}
    step_start = None
    last_step = None

    ########################################################
    # PROGRESS + TIMING HELPERS
    ########################################################

    def update(step, percent):
        if progress_callback:
            progress_callback(step, percent)
        else:
            print(step)

    def mark_step(name):
        nonlocal step_start, last_step

        if step_start is not None and last_step is not None:
            step_timings[last_step] = round(
                time.time() - step_start,
                2
            )

        step_start = time.time()
        last_step = name

    ########################################################
    # STEP 1 — LOAD DOCUMENTS
    ########################################################

    mark_step("Loading documents")
    update("STEP 1 — Loading documents", 5)

    sections = load_all_documents(document_folder)

    update(f"Loaded sections: {len(sections)}", 10)

    ########################################################
    # TRANSPARENCY LAYER
    ########################################################

    retrieval_sources = [
        {
            "document": s.get("document", "unknown"),
            "section_id": s.get("section_id"),
            "title": s.get("title"),
            "level": s.get("level"),
            "retrieval_text": s.get("retrieval_text", "")
        }
        for s in sections
    ]

    ########################################################
    # STEP 2 — BUILD RETRIEVAL SYSTEM
    ########################################################

    mark_step("Building retrieval system")
    update("STEP 2 — Building retrieval system", 20)

    retrieval_system = build_section_index(sections)

    ########################################################
    # STEP 3 — RETRIEVE RELEVANT SECTIONS (RAG FIRST)
    ########################################################

    mark_step("Retrieving relevant sections")
    update("STEP 3 — Retrieving sections", 35)

    relevant_sections = retrieve_sections(
        query=user_request,
        retrieval_system=retrieval_system,
        k=25
    )

    update(
        f"Retrieved sections: {len(relevant_sections)}",
        45
    )

    ########################################################
    # STEP 4 — EXTRACT REQUIREMENTS ONLY FROM
    # RETRIEVED SECTIONS
    ########################################################

    mark_step("Extracting requirements")
    update("STEP 4 — Extracting requirements", 55)

    retrieved_requirements = build_requirement_database(
        relevant_sections,
        llm_client
    )

    ########################################################
    # DEDUPLICATE REQUIREMENTS
    ########################################################

    retrieved_requirements = deduplicate_requirements(
        retrieved_requirements
    )

    update(
        f"Unique retrieved requirements: "
        f"{len(retrieved_requirements)}",
        65
    )

    ########################################################
    # STEP 5 — GENERATE PROCEDURE
    ########################################################

    mark_step("Generating procedure")
    update("STEP 5 — Generating procedure", 72)

    procedure_obj = generate_general_procedure(
        user_request=user_request,
        retrieval_system=retrieval_system,
        procedure_template=procedure_template,
        llm_client=llm_client
    )

    procedure = procedure_obj["procedure_text"]
    procedure_sections = procedure_obj["sections"]

    ########################################################
    # DEDUPLICATED GENERATION SOURCES
    ########################################################

    all_generation_sources = []
    seen = set()

    for section in procedure_sections:

        for src in section.get("sources", []):

            key = (
                src.get("document"),
                src.get("section_id"),
                src.get("title")
            )

            if key in seen:
                continue

            seen.add(key)
            all_generation_sources.append(src)

    ########################################################
    # STEP 6 — COVERAGE VALIDATION
    ########################################################

    mark_step("Requirement coverage validation")
    update("STEP 6 — Coverage validation", 82)

    coverage = validate_requirement_coverage(
        procedure,
        retrieved_requirements,
        llm_client
    )

    ########################################################
    # STEP 7 — PROCEDURE VALIDATION
    ########################################################

    mark_step("Enterprise validation")
    update("STEP 7 — Enterprise validation", 90)

    validation = validate_procedure(
        procedure,
        {
            "retrieved_requirements": retrieved_requirements
        },
        llm_client
    )

    ########################################################
    # STEP 8 — SAVE OUTPUTS
    ########################################################

    mark_step("Saving documents")
    update("STEP 8 — Saving documents", 97)

    output_path, output_path2 = save_procedure_docx(
        procedure=procedure_obj,
        validation_text=validation,
        all_generation_sources
        
    )

    ########################################################
    # FINALIZE TIMING
    ########################################################

    if step_start is not None and last_step is not None:
        step_timings[last_step] = round(
            time.time() - step_start,
            2
        )

    total_time = round(time.time() - start_time, 2)

    update(
        f"DONE — Completed in {total_time}s",
        100
    )

    ########################################################
    # RETURN RESULTS
    ########################################################

    return {
        "procedure": procedure,
        "coverage": coverage,
        "validation": validation,

        "output_path": output_path,
        "output_path_validation": output_path2,

        # retrieved RAG context
        "relevant_sections": relevant_sections,

        # requirements
        "retrieved_requirements": retrieved_requirements,

        # provenance
        "retrieval_sources": retrieval_sources,
        "generation_sources": all_generation_sources,

        # metadata
        "documents_loaded": len(sections),
        "total_retrieved_requirements": len(
            retrieved_requirements
        ),

        # analytics
        "timings": step_timings,
        "runtime_seconds": total_time,

        # template
        "procedure_template": procedure_template
    }

In [ ]:
#desktop app
import sys
import os

from PySide6.QtWidgets import (
    QApplication,
    QWidget,
    QLabel,
    QPushButton,
    QTextEdit,
    QVBoxLayout,
    QMessageBox,
    QComboBox,
    QProgressBar,
    QCheckBox,
    QLineEdit
)

from PySide6.QtCore import QThread, Signal

from pipeline import run_workflow
from llm_client import LLMClient


############################################################
# WORKER THREAD
############################################################

class WorkflowWorker(QThread):

    progress = Signal(str, int)
    finished = Signal(dict)
    error = Signal(str)

    def __init__(
        self,
        user_request,
        document_folder,
        llm_client,
        procedure_template
    ):
        super().__init__()

        self.user_request = user_request
        self.document_folder = document_folder
        self.llm_client = llm_client
        self.procedure_template = procedure_template

    def run(self):

        try:

            result = run_workflow(
                user_request=self.user_request,
                document_folder=self.document_folder,
                llm_client=self.llm_client,
                procedure_template=self.procedure_template,
                progress_callback=self.update_progress
            )

            self.finished.emit(result)

        except Exception as e:
            self.error.emit(str(e))

    def update_progress(self, message, percent):
        self.progress.emit(message, percent)


############################################################
# MAIN GUI
############################################################

class ProcedureGeneratorGUI(QWidget):

    def __init__(self):
        super().__init__()

        self.setWindowTitle("Enterprise Procedure Generator")
        #self.setGeometry(200, 200, 1200, 1000)
        self.setGeometry(200, 100, 1600, 1200)
        ####################################################
        # KNOWLEDGE BASE
        ####################################################

        self.kb_label = QLabel("Select Knowledge Base")

        self.kb_dropdown = QComboBox()

        self.load_knowledge_bases()

        ####################################################
        # LLM CONFIG
        ####################################################

        self.testing_mode = QCheckBox("Use Ollama (Testing Mode)")
        self.testing_mode.setChecked(True)

        self.api_key_input = QLineEdit()
        self.api_key_input.setPlaceholderText(
            "API Key (ignored in testing mode)"
        )

        self.base_url_input = QLineEdit()
        self.base_url_input.setPlaceholderText(
            "LLM Base URL (for production)"
        )

        self.model_input = QLineEdit()
        self.model_input.setPlaceholderText(
            "Model name (e.g. llama3)"
        )

        self.model_input.setText("llama3")

        ####################################################
        # USER REQUEST
        ####################################################

        self.prompt_label = QLabel("Procedure Request")

        self.prompt_input = QTextEdit()

        self.prompt_input.setPlaceholderText(
            "Example: Generate startup procedure for PX-400 pressure system"
        )

        ####################################################
        # PROCEDURE TEMPLATE (NEW)
        ####################################################

        self.template_label = QLabel(
            "Procedure Structure / Template"
        )

        self.template_input = QTextEdit()

        self.template_input.setPlaceholderText(
            """Example:

1. Objective
2. Scope
3. Preconditions
    3.1 Required Conditions
    3.2 Exceptions
4. Required Equipment
5. Safety Precautions
6. Procedure Steps
7. Validation Checks
8. Escalation Conditions
"""
        )

        self.template_input.setPlainText(
            """1. Objective
2. Scope
3. Preconditions
    3.1 Required Conditions
    3.2 Exceptions
4. Required Equipment
5. Safety Precautions
6. Procedure Steps
7. Validation Checks
8. Escalation Conditions"""
        )

        ####################################################
        # GENERATE BUTTON
        ####################################################

        self.generate_button = QPushButton(
            "Generate Procedure"
        )

        self.generate_button.clicked.connect(
            self.generate_procedure
        )

        ####################################################
        # PROGRESS
        ####################################################

        self.progress_bar = QProgressBar()
        self.progress_bar.setValue(0)

        self.status_label = QLabel("Status: Idle")

        ####################################################
        # LOG OUTPUT
        ####################################################

        self.log_label = QLabel("Execution Log")

        self.log_output = QTextEdit()
        self.log_output.setReadOnly(True)

        ####################################################
        # TIMING
        ####################################################

        self.timing_label = QLabel(
            "Step Timing Analytics"
        )

        self.timing_output = QTextEdit()
        self.timing_output.setReadOnly(True)

        ####################################################
        # RETRIEVAL
        ####################################################

        self.retrieval_label = QLabel(
            "Retrieval Transparency"
        )

        self.retrieval_output = QTextEdit()
        self.retrieval_output.setReadOnly(True)

        ####################################################
        # OUTPUT
        ####################################################

        #self.procedure_label = QLabel(
        #    "Generated Procedure"
        #)

        #self.procedure_output = QTextEdit()
        #self.procedure_output.setReadOnly(True)

        #self.validation_label = QLabel("Validation")

        #self.validation_output = QTextEdit()
        #self.validation_output.setReadOnly(True)

        ####################################################
        # LAYOUT
        ####################################################

        layout = QVBoxLayout()

        layout.addWidget(self.kb_label)
        layout.addWidget(self.kb_dropdown)

        ####################################################
        # LLM SETTINGS
        ####################################################

        layout.addWidget(self.testing_mode)
        layout.addWidget(self.api_key_input)
        layout.addWidget(self.base_url_input)
        layout.addWidget(self.model_input)

        ####################################################
        # USER INPUT
        ####################################################

        layout.addWidget(self.prompt_label)
        #layout.addWidget(self.prompt_input)
        layout.addWidget(self.prompt_input, 2)

        ####################################################
        # TEMPLATE INPUT
        ####################################################

        layout.addWidget(self.template_label)
        #layout.addWidget(self.template_input)
        layout.addWidget(self.template_input, 15)

        ####################################################
        # GENERATE
        ####################################################

        layout.addWidget(self.generate_button)

        ####################################################
        # PROGRESS
        ####################################################

        layout.addWidget(self.progress_bar)
        layout.addWidget(self.status_label)

        ####################################################
        # LOGS
        ####################################################

        layout.addWidget(self.log_label)
        #layout.addWidget(self.log_output)
        layout.addWidget(self.log_output, 16)

        ####################################################
        # TIMINGS
        ####################################################

        layout.addWidget(self.timing_label)
        #layout.addWidget(self.timing_output)
        layout.addWidget(self.timing_output, 16)

        ####################################################
        # RETRIEVAL
        ####################################################

        layout.addWidget(self.retrieval_label)
        #layout.addWidget(self.retrieval_output)
        layout.addWidget(self.retrieval_output, 10)

        ####################################################
        # OUTPUTS
        ####################################################

        #layout.addWidget(self.procedure_label)
        #layout.addWidget(self.procedure_output)

        #layout.addWidget(self.validation_label)
        #layout.addWidget(self.validation_output)

        self.setLayout(layout)

    ############################################################
    # LOAD KNOWLEDGE BASES
    ############################################################

    def load_knowledge_bases(self):

        kb_root = "knowledge_bases"

        if not os.path.exists(kb_root):
            os.makedirs(kb_root)

        folders = [
            f for f in os.listdir(kb_root)
            if os.path.isdir(os.path.join(kb_root, f))
        ]

        self.kb_dropdown.addItems(folders)

    ############################################################
    # GENERATE PROCEDURE
    ############################################################

    def generate_procedure(self):

        user_request = self.prompt_input.toPlainText().strip()

        if not user_request:

            QMessageBox.warning(
                self,
                "Missing Input",
                "Enter a request."
            )

            return

        ####################################################
        # PROCEDURE TEMPLATE
        ####################################################

        procedure_template = (
            self.template_input.toPlainText().strip()
        )

        if not procedure_template:

            QMessageBox.warning(
                self,
                "Missing Template",
                "Enter a procedure structure."
            )

            return

        ####################################################
        # KNOWLEDGE BASE
        ####################################################

        selected_kb = self.kb_dropdown.currentText()

        document_folder = os.path.join(
            "knowledge_bases",
            selected_kb
        )

        ####################################################
        # CREATE LLM CLIENT
        ####################################################

        llm_client = LLMClient(
            api_key=self.api_key_input.text() or None,
            base_url=self.base_url_input.text() or None,
            model=self.model_input.text() or "llama3",
            testing=self.testing_mode.isChecked()
        )

        ####################################################
        # RESET UI
        ####################################################

        self.generate_button.setEnabled(False)

        self.progress_bar.setValue(0)

        self.log_output.clear()
        self.timing_output.clear()
        self.retrieval_output.clear()

        #self.procedure_output.clear()
        #self.validation_output.clear()

        self.status_label.setText("Status: Starting...")

        ####################################################
        # START WORKER
        ####################################################

        self.worker = WorkflowWorker(
            user_request=user_request,
            document_folder=document_folder,
            llm_client=llm_client,
            procedure_template=procedure_template
        )

        self.worker.progress.connect(
            self.update_progress
        )

        self.worker.finished.connect(
            self.on_finished
        )

        self.worker.error.connect(
            self.on_error
        )

        self.worker.start()

    ############################################################
    # LIVE PROGRESS
    ############################################################

    def update_progress(self, message, percent):

        self.progress_bar.setValue(percent)

        self.status_label.setText(
            f"Status: {message}"
        )

        self.log_output.append(message)

    ############################################################
    # FINISHED
    ############################################################

    def on_finished(self, result):

        self.generate_button.setEnabled(True)

        ####################################################
        # MAIN OUTPUTS
        ####################################################

        #self.procedure_output.setPlainText(
        #    result["procedure"]
        #)

        #self.validation_output.setPlainText(
        #    result["validation"]
        #)

        ####################################################
        # TIMING ANALYTICS
        ####################################################

        timings = result.get("timings", {})

        timing_text = "STEP TIMING BREAKDOWN:\n\n"

        total = 0

        for step, t in timings.items():

            timing_text += f"{step}: {t}s\n"

            total += t

        timing_text += (
            f"\nTOTAL PIPELINE TIME: "
            f"{round(total, 2)}s"
        )

        self.timing_output.setPlainText(
            timing_text
        )

        ####################################################
        # RETRIEVAL TRANSPARENCY
        ####################################################

        sources = result.get(
            "retrieval_sources",
            []
        )

        retrieval_text = (
            "DOCUMENT SOURCES USED:\n\n"
        )

        for s in sources:
            retrieval_text += (
                f"- {s.get('document')} | "
                f"{s.get('section_id')} | "
                f"{s.get('title')}\n"
            )

        self.retrieval_output.setPlainText(
            retrieval_text
        )

        ####################################################
        # FINAL STATUS
        ####################################################

        self.status_label.setText(
            "Status: Completed"
        )

        QMessageBox.information(
            self,
            "Success",
            "Procedure generated successfully!"
        )
        ####################################################
        # PROVENANCE (NEW — IMPORTANT)
        ####################################################
        
        provenance = result.get("provenance_map", {})
        
        if provenance:
        
            prov_text = "REQUIREMENT PROVENANCE:\n\n"
        
            for req, meta in list(provenance.items())[:50]:
        
                prov_text += (
                    f"- {req[:80]}...\n"
                    f"  Document: {meta.get('document')}\n"
                    f"  Section: {meta.get('section_id')} | {meta.get('title')}\n"
                    f"  Type: {meta.get('type')} | Phase: {meta.get('phase')}\n\n"
                )
        
            self.retrieval_output.append("\n\n" + prov_text)

    ############################################################
    # ERROR HANDLING
    ############################################################

    def on_error(self, error):

        self.generate_button.setEnabled(True)

        self.status_label.setText(
            "Status: Error"
        )

        QMessageBox.critical(
            self,
            "Error",
            error
        )


############################################################
# RUN APP
############################################################

if __name__ == "__main__":

    app = QApplication(sys.argv)

    app.setStyleSheet("""
    QWidget {
        background-color: #1e1e1e;
        color: white;
        font-size: 14px;
    }

    QPushButton {
        background-color: #0078d4;
        padding: 8px;
    }

    QTextEdit {
        background-color: #2b2b2b;
    }

    QLineEdit {
        background-color: #2b2b2b;
        padding: 6px;
    }

    QComboBox {
        background-color: #2b2b2b;
        padding: 6px;
    }

    QProgressBar {
        border: 1px solid #444;
        text-align: center;
    }
    """)

    window = ProcedureGeneratorGUI()

    window.show()

    sys.exit(app.exec())

In [ ]:
#llm client
import requests

try:
    import ollama
except ImportError:
    ollama = None


class LLMClient:
    """
    Generic LLM client with:
    - Ollama local testing mode
    - API-based production mode
    """

    def __init__(self, api_key=None, base_url=None, model=None, testing=True):

        self.api_key = api_key
        self.base_url = base_url.rstrip("/") if base_url else None
        self.model = model
        self.testing = testing

    # ----------------------------------------------------
    # MAIN ENTRY
    # ----------------------------------------------------

    def chat(self, prompt):

        if self.testing:
            return self._chat_ollama(prompt)

        return self._chat_api(prompt)

    # ----------------------------------------------------
    # LOCAL OLLAMA MODE
    # ----------------------------------------------------
    def generate(self, prompt):
        return self.chat(prompt)
    def _chat_ollama(self, prompt):

        if ollama is None:
            raise ImportError("ollama package not installed")

        response = ollama.chat(
            model=self.model or "llama3",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        return response["message"]["content"]

    # ----------------------------------------------------
    # API MODE (enterprise / internal LLM)
    # ----------------------------------------------------

    def _chat_api(self, prompt):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": self.model,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }

        response = requests.post(
            f"{self.base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        return data["choices"][0]["message"]["content"]